## Initialize Environment

All Python packages used are listed in `environment.yml`. The following commands create and activate the `sla` environment:

```bash
conda env create -f environment.yml
conda activate sla
```

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

OUTPUT_DIR = Path("build/demo")
RUN_FIXED_POLICY = False
RUN_EXPENSIVE_BO = False

## Reproduce the Paper Outputs

The default, fast reproduction uses the authenticated aggregate files in `paper_data/`. It rebuilds the manuscript figures and tables without rerunning Bayesian optimization or reading request-level records. The displayed map is rebuilt from compact tract costs and the tracked census shapefile; it does not require the legacy tract-linkage file.

The optional simulation cells below use the archived December 1, 2023 snapshots of [Forestry Service Requests](https://data.cityofnewyork.us/Environment/Forestry-Service-Requests/mu46-p9is/about_data) and [Forestry Inspections](https://data.cityofnewyork.us/Environment/Forestry-Inspections/4pt5-3vv4/about_data). See `data_raw/README.md` before setting either run flag to `True`.

In [ ]:
completed = subprocess.run(
    [sys.executable, "-m", "analysis.reproduce",
     "--data-dir", "paper_data", "--output-dir", str(OUTPUT_DIR)],
    check=True, capture_output=True, text=True,
)
generated_outputs = [Path(line) for line in completed.stdout.splitlines() if line.strip()]
print(f"Generated {len(generated_outputs)} outputs under {OUTPUT_DIR}.")

In [ ]:
policy_means = pd.read_csv("paper_data/primary/inference_means.csv")
policy_means.loc[
    policy_means["display_roles"].notna(),
    ["budget_group", "display_roles", "efficiency_pct_historical",
     "equity_pct_historical", "n_seeds"],
].reset_index(drop=True)

## Running Bayesian Optimization Loop

The file `run_bo.py` contains code needed to perform the Bayesian optimization routine; the optimization problem is defined in `bo_problem.py`, and the simulator on which the optimization is done is defined in `simulator.py`.

The optional cell below illustrates a small Borough-budget search. It uses the primary manuscript setting: a 100-day noninspection penalty (`--drop_cost 100`), $\rho=0.15$, median inspection delay (`--obj delay_median`), and the all-request geographic-equity objective (`--equity allrequest`). Set `RUN_EXPENSIVE_BO = True` only after restoring the raw inputs through Git LFS. The full paper searches were substantially larger; their compact evaluated outputs are already reproduced above.

`run_bo.py` writes local checkpoints, so this step is intentionally skipped by default when the notebook is run from top to bottom.

In [ ]:
if RUN_EXPENSIVE_BO:
    from evaluate_selected_policy import preflight_data
    preflight_data(Path.cwd())
    subprocess.run(
        [sys.executable, "run_bo.py", "--batch_size", "4",
         "--n_steps", "2", "--city_budget", "0",
         "--drop_cost", "100", "--rho", "0.15",
         "--obj", "delay_median", "--equity", "allrequest"],
        check=True,
    )
else:
    print("Skipped Bayesian optimization; set RUN_EXPENSIVE_BO = True to run it.")

# Evaluating a Selected Policy

The current locked policy parameters are public compact inputs. By default, the cells below display their authenticated 25-simulation summaries. Set `RUN_FIXED_POLICY = True` to rerun the Borough most-equitable policy for seed 321; this requires either the authenticated merged cache or the two full raw snapshots described in `data_raw/README.md`.

In [ ]:
from evaluate_selected_policy import compare, evaluate, load_policy, preflight_data

ROLE = "borough_most_equitable"
policy = load_policy(
    Path("paper_data/primary/selected_policy_parameters.csv"), ROLE
)
{key: policy[key] for key in ("role", "policy_id", "vector_hash", "budget_group")}

In [ ]:
policy_parameters = pd.read_csv("paper_data/primary/selected_policy_parameters.csv")
policy_parameters.loc[policy_parameters["role"].eq(ROLE)].head(6)

In [ ]:
scores_path = Path("paper_data/primary/selected_policy_scores_by_seed.csv")
if RUN_FIXED_POLICY:
    input_mode = preflight_data(Path.cwd())
    result = evaluate(Path.cwd(), policy, seed=321, year=2019)
    result["input_mode"] = input_mode
    result.update(compare(result, scores_path))
else:
    checked_scores = pd.read_csv(scores_path)
    result = checked_scores.loc[
        checked_scores["policy_id"].eq(policy["policy_id"])
        & checked_scores["random_seed"].eq(321)
        & checked_scores["calendar_year"].eq(2019)
    ].iloc[0].to_dict()
    result["status"] = "checked-in result; no simulation run"
result

In [ ]:
cell_outcomes = pd.read_csv("paper_data/primary/selected_policy_cells.csv")
cell_outcomes.loc[
    cell_outcomes["role"].eq(ROLE),
    ["borough", "category", "sla_days_mean",
     "inspection_fraction_mean", "n_evaluations"],
].reset_index(drop=True)

In [ ]:
map_path = OUTPUT_DIR / "figures" / "allrequest_cost_maps_main.png"
map_path

In [ ]:
if not map_path.is_file():
    raise FileNotFoundError(f"Compact reproduction did not create {map_path}")
display(Image(filename=str(map_path)))